# Driver Distraction Detection YOLOv8 Training

Welcome! This notebook will guide you through training your driver distraction detection model on a **free GPU** in Google Colab.

### ⚠️ Crucial Step: Enable GPU
Before running any cells, ensure your runtime is using a GPU:
1. In the top menu, click **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (or any GPU) from the Hardware Accelerator dropdown.
3. Click **Save**.

## 1. Install Ultralytics

In [ ]:
!pip install ultralytics

## 2. Prepare the Dataset
Choose **one** of the two options below to load your dataset.

### Option A: Upload from your Computer (Zipped)
1. Zip your local dataset folder (select `train`, `valid`, `test` folders and `data.yaml`, then compress them into `Distract.zip`).
2. Click the folder icon on the left sidebar in Colab.
3. Drag and drop `Distract.zip` into the file explorer area.
4. Run the cell below to unzip it.

In [ ]:
# Unzip your uploaded dataset
!mkdir -p /content/Distract
!unzip -q /content/Distract.zip -d /content/Distract
print("Dataset unzipped successfully!")

### Option B: Download directly from Roboflow (No Upload Needed)
If you want to pull directly from Roboflow, run the following commands. You'll need your Roboflow Private API Key.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
# Replace 'YOUR_API_KEY' with your actual Roboflow API key
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("aiinoimind").project("car-driving-analysis")
dataset = project.version(1).download("yolov8")

## 3. Configure Dataset Paths for Colab
We need to make sure the dataset's `data.yaml` file points to the Colab paths.

In [ ]:
import yaml
import os

# Find the data.yaml file path depending on how you loaded the dataset
yaml_path = "/content/Distract/data.yaml"

if not os.path.exists(yaml_path):
    # If using Roboflow download, search for it
    for root, dirs, files in os.walk("/content"):
        if "data.yaml" in files:
            yaml_path = os.path.join(root, "data.yaml")
            break

print(f"Using dataset config: {yaml_path}")

with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Set path to where images actually are in Colab
config['path'] = os.path.dirname(yaml_path)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

print("data.yaml configured successfully!")
print(config)

## 4. Train the Model
We will train a YOLOv8n (nano) model. It is fast, accurate, and extremely efficient for live-streaming on a laptop/CPU.

In [ ]:
from ultralytics import YOLO

# Load pretrained nano model
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=512,
    batch=16,
    device=0, # GPU 0
    project='driver_distraction',
    name='yolov8n_colab'
)

## 5. Download the Trained Model Weights
Run this cell to download the trained `best.pt` file directly to your local computer. Place this file in your project folder so `detect_live.py` can load it!

In [ ]:
from google.colab import files
import os

weights_path = '/content/driver_distraction/yolov8n_colab/weights/best.pt'

if os.path.exists(weights_path):
    print("Downloading best.pt weights...")
    files.download(weights_path)
else:
    print(f"Weights file not found at {weights_path}. Check your training run results!")